# 第九课:Transformer Block——总装车间

讲义:[09-transformer-block.md](09-transformer-block.md)。残差 + Pre-LN + 两个注入的子层。

In [ ]:
import numpy as np, math

def softmax_rows(M):
    m = M.max(axis=-1, keepdims=True); e = np.exp(M - m)
    return e / e.sum(-1, keepdims=True)

def ln(x, eps=1e-5):
    return (x - x.mean(-1, keepdims=True)) / np.sqrt(x.var(-1, keepdims=True) + eps)

rng = np.random.default_rng(42)
n, d = 4, 8
Wq, Wk, Wv, Wo = (rng.normal(size=(d, d)) * 0.2 for _ in range(4))

mask = np.triu(np.ones((n, n), bool), 1)      # causal: 上三角遮住

def attn(x):
    Q, K, V = x @ Wq, x @ Wk, x @ Wv
    s = Q @ K.T / math.sqrt(d)
    s = np.where(mask, -np.inf, s)
    return softmax_rows(s) @ V @ Wo

def ffn_placeholder(x):
    return x @ rng.normal(size=(d, d)) * 0.1   # 第 10 课换正式版

def make_block(attn, ffn):
    def block(x):
        x = x + attn(ln(x))     # 子层1 + 残差高速
        x = x + ffn(ln(x))      # 子层2 + 残差高速
        return x
    return block

block = make_block(attn, ffn_placeholder)
X = rng.normal(size=(n, d))
out = block(X)
print('shape 不变:', X.shape, '→', out.shape)
print('|x| =', round(np.linalg.norm(X[0]), 3), '→ 子层1后 =', round(np.linalg.norm((X + attn(ln(X)))[0]), 3))

## 注意力在传递信息:扰动传播实验(讲义 §3)

In [ ]:
# 自包含实验:固定 FFN 权重(消除随机污染),非常量扰动
import numpy as np, math

def softmax_rows(M):
    m = M.max(axis=-1, keepdims=True); e = np.exp(M - m)
    return e / e.sum(-1, keepdims=True)

def ln(x, eps=1e-5):
    return (x - x.mean(-1, keepdims=True)) / np.sqrt(x.var(-1, keepdims=True) + eps)

rng2 = np.random.default_rng(42)
n, d = 4, 8
Wq, Wk, Wv, Wo = (rng2.normal(size=(d, d)) * 0.2 for _ in range(4))
W_ffn = rng2.normal(size=(d, d)) * 0.1                  # 固定的占位 FFN
mask = np.triu(np.ones((n, n), bool), 1)

def attn2(x):
    Q, K, V = x @ Wq, x @ Wk, x @ Wv
    s = np.where(mask, -np.inf, Q @ K.T / math.sqrt(d))
    return softmax_rows(s) @ V @ Wo

def block2(x):
    x = x + attn2(ln(x))
    x = x + (ln(x) @ W_ffn)
    return x

X = rng2.normal(size=(n, d))
pert = np.zeros(d); pert[0] = 3.0; pert[1] = -1.5        # 非常量扰动

X2 = X.copy(); X2[0] += pert
print('扰动词1(非常量) -> 词4 变化范数:', round(float(np.linalg.norm(block2(X2)[3] - block2(X)[3])), 4), ' <- 信息流动了')
X3 = X.copy(); X3[3] += pert
print('扰动词4(非常量) -> 词1 变化范数:', round(float(np.linalg.norm(block2(X3)[0] - block2(X)[0])), 12), ' <- causal: 看不到未来')
X4 = X.copy(); X4[0] += np.full(d, 3.0)
print('整行+3(常向量)  -> 词4 变化范数:', round(float(np.linalg.norm(block2(X4)[3] - block2(X)[3])), 12), ' <- 被 LN 平移不变性吸收!')

## Pre-LN vs Post-LN 范数对比(讲义 §4)

In [ ]:
pre  = X + attn(ln(X))       # Pre-LN: 主干保留原尺度
post = ln(X + attn(X))       # Post-LN: 每层被 LN 压回固定尺度
print('Pre-LN  子层输出范数:', round(np.linalg.norm(pre[0]), 3))
print('Post-LN 子层输出范数:', round(np.linalg.norm(post[0]), 3))

## 堆 N 层(第 20 课 Mini-GPT 的雏形)

形状不变 ⇒ `for _ in range(N): x = block(x)`。注意:这里占位 FFN 每次调用重新随机——真实模型权重固定,第 20 课修正。

In [ ]:
W_ffn = rng.normal(size=(d, d)) * 0.1            # 固定占位权重
def ffn_fixed(x):
    return x @ W_ffn
block_fixed = make_block(attn, ffn_fixed)
x = X.copy()
for i in range(4):
    x = block_fixed(x)
    print(f'第 {i+1} 层后 主干范数(第1词) = {np.linalg.norm(x[0]):.3f}')

## 作业现场

- C2:独立复现扰动传播两个方向(非零 / 恰为零)。
- C3:写 Post-LN 版 make_block_post,复现范数对比。
- D1:写逐层打印主干范数的检查代码。
- E1:残差分支缩放 0.5,观察范数增长。

In [ ]:
# TODO(C3): Post-LN 版
def make_block_post(attn, ffn):
    # 你的代码:x = ln(x + attn(x)); x = ln(x + ffn(x))
    ...

# TODO(E1): 缩放残差分支
def make_block_scaled(attn, ffn, alpha=0.5):
    # 你的代码:x = x + alpha * attn(ln(x)) ...
    ...

# 观察:for i in range(8): 打印两种 block 的主干范数增长曲线